In [1]:
# %%capture --no-stderr
# !pip install -U langgraph
# !pip install psycopg2-binary==2.9.9
# !pip install python-dotenv

In [2]:
from pprint import pprint

In [3]:
import os
import sys
import boto3
import json
import logging
import mlflow
from mlflow.entities import SpanType


from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [4]:
mlflow.__version__

'2.20.3'

In [5]:
# Get the absolute path to the backend directory
current_dir = os.getcwd()  # Get current working directory
backend_path = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(backend_path)

# Load environment variables from .env file
load_dotenv()

True

In [6]:
from backend import app

In [7]:
# # Enable auto-tracing for Amazon Bedrock
mlflow.bedrock.autolog()
mlflow.langchain.autolog()

2025/03/25 18:04:33 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


In [8]:
try:
    mlflow_arn = os.getenv('MLFLOW_TRACKING_ARN')
    if mlflow_arn:
        print(f'\nMLFlow tracking URI: {mlflow_arn}')
        mlflow.set_tracking_uri(mlflow_arn)
        print('Successfully connected to MLFlow tracking server')
except Exception as e:
    print('Error connecting to MLFlow tracking server')
    raise e


MLFlow tracking URI: arn:aws:sagemaker:us-west-2:577201992296:mlflow-tracking-server/bedrock-chatbot-mlflow
Successfully connected to MLFlow tracking server


In [9]:
import uuid
session_id = str(uuid.uuid4())

In [10]:
session_id

'd454926c-761f-47a3-b0be-ccf2c6d45f08'

In [11]:
mlflow.set_experiment(str(session_id))
mlflow.tracing.enable_notebook_display()

2025/03/25 18:04:34 INFO mlflow.tracking.fluent: Experiment with name 'd454926c-761f-47a3-b0be-ccf2c6d45f08' does not exist. Creating a new experiment.


In [12]:
messages = "I need to cancel an order but don't remember my order id"

In [13]:
session_dict = {'session_id': session_id}

In [14]:
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <============== Node: entry_intent ===================> 



"<reply>Hello! I'll be happy to help you find and check your order. To locate your order, I'll need some information to identify your account first. Could you please provide either your email address, phone number, or username associated with your account?</reply>"

In [15]:
messages = "My phone number is 808-555-6149"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <============== Node: entry_intent ===================> 


 ------- RETRY 1 with delay 1.01s ---------- 


 <=============== Node: order_confirmation =============> 



"<reply>Hello Ana! I can see you have two recent orders:\n\n1. Order #81695: Wireless Keyboard - Status: Processing\n2. Order #86420: Fitness Tracker - Status: Delivered\n\nSince only processing orders can be cancelled, I can help you cancel Order #81695 for the Wireless Keyboard if that's the one you'd like to cancel. Would you like me to proceed with cancelling this order?</reply>"

In [16]:
messages = "I would like to cancel 81695"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <=============== Node: order_confirmation =============> 


 ------- RETRY 1 with delay 1.08s ---------- 


 ------- RETRY 1 with delay 1.01s ---------- 



'<reply>Just to confirm - you want to cancel Order #81695 for the Wireless Keyboard priced at $49.99? Please confirm with a yes if this is correct.</reply>'

In [17]:
messages = "That's correct!"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <=============== Node: order_confirmation =============> 


 ------- RETRY 1 with delay 1.09s ---------- 


 <=============== Node: resolution =====================> 


 ------- RETRY 1 with delay 1.02s ---------- 


 ------- CANCEL ORDER: 81695 ---------- 


 ------- QUERY: UPDATE orders SET status = 'Cancelled' WHERE id = %s AND status = 'Processing' ----------- 

Cancel successful: Mockup function not real implementation


"<reply>I've successfully cancelled your order #81695 for the Wireless Keyboard. You should receive a confirmation email shortly. Is there anything else I can help you with?</reply>"

In [18]:
messages = "I'm all set. Thank you"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <=============== Node: resolution =====================> 


 ------- RETRY 1 with delay 1.03s ---------- 



"<thinking>\n1. The user has indicated they don't need any further assistance\n2. I should provide a polite closing response\n</thinking>\n\n<reply>You're welcome! Thank you for contacting AnyCompany customer service. Have a great day!</reply>"